# 作业 4：序列模型、RNN 与 注意力机制

**学生姓名**：傅昱翔 
**学号**：20234080328  
**日期**：2026年6月19日

## 2 序列模型

### 2.1 理论计算题

**题目**：给定字符串序列 "ababc"。词汇表为 {'a','b','c'}。假设采用一阶马尔可夫模型，使用拉普拉斯平滑（加 1 平滑）估计 $p('a' | 'b')$ 和 $p('c' | 'b')$。

**解答**：

1. **统计转移频数**：
   序列 "ababc" 中的相邻对为：(a,b), (b,a), (a,b), (b,c)。
   - 以 'b' 开头的转移：
     - 'b' -> 'a': 出现 1 次
     - 'b' -> 'c': 出现 1 次
     - 'b' -> 'b': 出现 0 次
   - 统计量：$count('b') = 2$ (作为前驱词出现的次数)。

2. **拉普拉斯平滑公式**：
   $P(x_t | x_{t-1}) = \frac{count(x_{t-1}, x_t) + 1}{count(x_{t-1}) + V}$，其中 $V = 3$（词汇表大小）。

3. **计算概率**：
   - $p('a' | 'b') = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$
   - $p('c' | 'b') = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$

### 2.2 编程题

实现 `preprocess_text` 函数。

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    # 1. 转换为小写，去除标点（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    tokens = text.split()
    
    # 3. 构造词汇表（按频率排序）
    counts = Counter(tokens)
    # 按频率降序排序，频率相同时按字母顺序排序以保证结果稳定
    sorted_tokens = sorted(counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {token: i for i, (token, _) in enumerate(sorted_tokens)}
    
    # 4. 生成特征序列和标签
    features = []
    labels = []
    for i in range(len(tokens) - n + 1):
        window = tokens[i : i + n]
        features.append(window)
        if i + n < len(tokens):
            labels.append(tokens[i + n])
        else:
            labels.append(None)
            
    return vocab, (features, labels)

# 测试示例
text_input = "The time machine"
n_val = 2
vocab, (features, labels) = preprocess_text(text_input, n_val)

print("词汇表:", vocab)
print("特征序列:", features)
print("标签列表:", labels)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征序列: [['the', 'time'], ['time', 'machine']]
标签列表: ['machine', None]


## 3 循环神经网络

### 3.1 理论计算题

**推导损失对 $W_{hh}$ 的梯度**：

给定：
$h_t = W_{hh} h_{t-1} + W_{hx} x_t$
$o_t = W_{oh} h_t$
$L = \frac{1}{2} \sum_{t=1}^T (o_t - y_t)^2$

对于总损失 $L$，其对 $W_{hh}$ 的梯度为：
$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \frac{\partial L_t}{\partial W_{hh}}$

利用链式法则展开第 $t$ 步的损失梯度：
$\frac{\partial L_t}{\partial W_{hh}} = \sum_{k=1}^t \frac{\partial L_t}{\partial o_t} \frac{\partial o_t}{\partial h_t} \left( \prod_{j=k+1}^t \frac{\partial h_j}{\partial h_{j-1}} \right) \frac{\partial h_k}{\partial W_{hh}}$

由于是线性 RNN：
- $\frac{\partial L_t}{\partial o_t} = (o_t - y_t)$
- $\frac{\partial o_t}{\partial h_t} = W_{oh}$
- $\frac{\partial h_j}{\partial h_{j-1}} = W_{hh}$
- $\frac{\partial h_k}{\partial W_{hh}} = h_{k-1}^\top$

代入得：
$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T (o_t - y_t) W_{oh} \sum_{k=1}^t (W_{hh})^{t-k} h_{k-1}^\top$

**梯度消失/爆炸的条件**：
梯度表达式中包含 $(W_{hh})^{t-k}$。其性质取决于 $W_{hh}$ 的最大特征值（谱半径）$\rho(W_{hh})$：
- **梯度爆炸**：若 $\rho(W_{hh}) > 1$，随着时间步 $t$ 增加，项 $(W_{hh})^{t-k}$ 会呈指数级增长。
- **梯度消失**：若 $\rho(W_{hh}) < 1$，随着时间步 $t$ 增加，项 $(W_{hh})^{t-k}$ 会迅速趋于 0，导致远距离的依赖无法学习。

### 3.2 编程题

实现单步 RNN 前向和反向传播。

In [2]:
import torch

def rnn_step_forward_backward(x_t, h_prev, W_xh, W_hh, b_h, dh_next):
    # 前向传播
    # h_t = tanh(x_t * W_xh + h_prev * W_hh + b_h)
    # 注意：通常线性映射后接激活函数
    z_t = torch.matmul(x_t, W_xh) + torch.matmul(h_prev, W_hh) + b_h
    h_t = torch.tanh(z_t)
    
    # 反向传播
    # 1. tanh 的梯度: dtanh(z)/dz = 1 - tanh^2(z) = 1 - h_t^2
    dz_t = dh_next * (1 - h_t**2)
    
    # 2. 计算各参数梯度
    # z_t = x_t @ W_xh + h_prev @ W_hh + b_h
    dx_t = torch.matmul(dz_t, W_xh.t())
    dh_prev = torch.matmul(dz_t, W_hh.t())
    
    # 权重梯度需要考虑 batch 的累加 (假设 batch 在第0维)
    dW_xh = torch.matmul(x_t.t(), dz_t)
    dW_hh = torch.matmul(h_prev.t(), dz_t)
    db_h = torch.sum(dz_t, dim=0)
    
    return h_t, dx_t, dh_prev, dW_xh, dW_hh, db_h

# 测试
batch_size, input_size, hidden_size = 2, 3, 4
x_t = torch.randn(batch_size, input_size)
h_prev = torch.randn(batch_size, hidden_size)
W_xh = torch.randn(input_size, hidden_size)
W_hh = torch.randn(hidden_size, hidden_size)
b_h = torch.randn(hidden_size)
dh_next = torch.randn(batch_size, hidden_size)

h_t, dx_t, dh_prev, dW_xh, dW_hh, db_h = rnn_step_forward_backward(x_t, h_prev, W_xh, W_hh, b_h, dh_next)
print("h_t shape:", h_t.shape)
print("dW_hh shape:", dW_hh.shape)

h_t shape: torch.Size([2, 4])
dW_hh shape: torch.Size([4, 4])


## 4 高级循环神经网络

### 4.1 理论计算题

**题目**：计算 $L$ 层深度双向 RNN 的参数总量。每层隐藏单元 $H$，输入 $D$，输出 $O$。

**解答**：
双向 RNN 每一层包含一个前向 RNN 和一个后向 RNN。

1. **第一层 (Layer 1)**：
   - 前向：$W_{xh}^{(1)} \in \mathbb{R}^{D \times H}, W_{hh}^{(1)} \in \mathbb{R}^{H \times H}, b_h^{(1)} \in \mathbb{R}^H$。参数数：$DH + H^2 + H$。
   - 后向：同上，参数数：$DH + H^2 + H$。
   - 第一层总计：$2(DH + H^2 + H)$。

2. **后续层 (Layer $l=2 \dots L$)**：
   - 输入是上一层双向输出的拼接，维度为 $2H$。
   - 前向：$W_{xh}^{(l)} \in \mathbb{R}^{2H \times H}, W_{hh}^{(l)} \in \mathbb{R}^{H \times H}, b_h^{(l)} \in \mathbb{R}^H$。参数数：$2H^2 + H^2 + H = 3H^2 + H$。
   - 后向：同上，参数数：$3H^2 + H$。
   - 每一层总计：$2(3H^2 + H) = 6H^2 + 2H$。

3. **输出层 (Output Layer)**：
   - 从最后一层拼接后的状态 $2H$ 映射到 $O$。
   - $W_{ho} \in \mathbb{R}^{2H \times O}, b_o \in \mathbb{R}^O$。参数数：$2HO + O$。

**总参数量表达式**：
$$\text{Total} = 2(DH + H^2 + H) + (L-1)(6H^2 + 2H) + 2HO + O$$

### 4.2 编程题

实现双向 RNN 编码器。

In [3]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(BiRNNEncoder, self).__init__()
        # 使用 torch.nn.RNN 实现双向 RNN
        self.rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim, 
                          num_layers=1, bidirectional=True)
        
    def forward(self, X):
        # X 形状: (seq_len, batch, input_dim)
        output, h_n = self.rnn(X)
        
        # output 形状: (seq_len, batch, 2 * hidden_dim)
        # 包含每个时间步拼接后的前向和后向隐藏状态
        
        # 最终时刻的拼接隐藏状态：
        # 对于双向 RNN，h_n 的形状是 (num_layers * num_directions, batch, hidden_dim)
        # 最终时刻的拼接即 h_n[0] (前向最后一步) 和 h_n[1] (后向最后一步，即序列开头)
        final_state = torch.cat([h_n[0], h_n[1]], dim=-1)
        
        return output, final_state

# 测试
seq_len, batch, input_dim, hidden_dim = 5, 2, 8, 16
X = torch.randn(seq_len, batch, input_dim)
model = BiRNNEncoder(input_dim, hidden_dim)
output, final_state = model(X)

print("Output shape:", output.shape) # 应为 (5, 2, 32)
print("Final state shape:", final_state.shape) # 应为 (2, 32)

Output shape: torch.Size([5, 2, 32])
Final state shape: torch.Size([2, 32])


## 5 嵌入向量

### 5.1 理论计算题

**Skip-gram 负采样目标函数**：

对于中心词 $c$ 和目标词 $o$，以及 $K$ 个负样本 $n_1, \dots, n_K$：
$$L = -\log \sigma(u_o^\top v_c) - \sum_{i=1}^K \log \sigma(-u_{n_i}^\top v_c)$$
其中 $v_c$ 是中心词向量，$u$ 是上下文词向量，$\sigma(x) = 1/(1+e^{-x})$ 是 sigmoid 函数。

**负采样方法**：
负样本从噪声分布 $P(w)$ 中采样。为了提升低频词的采样概率，通常使用词频 $f(w)$ 的 $3/4$ 次方：
$$P(w) = \frac{f(w)^{0.75}}{\sum_{j=1}^V f(w_j)^{0.75}}$$

### 5.2 编程题

实现 CBOW 前向传播与损失计算。

In [4]:
import torch
import torch.nn.functional as F

def cbow_forward_loss(context_indices, target_index, W_in, W_out):
    # context_indices: (context_size,) 包含上下文词的 ID
    # W_in: (V, d)
    # W_out: (d, V)
    
    # 1. 获取上下文词向量并取平均
    # context_vectors: (context_size, d)
    context_vectors = W_in[context_indices]
    h = torch.mean(context_vectors, dim=0, keepdim=True) # (1, d)
    
    # 2. 计算输出层的得分 (logits)
    # output: (1, V)
    logits = torch.matmul(h, W_out)
    
    # 3. 计算交叉熵损失
    # target_index 需要包装成 tensor
    target = torch.tensor([target_index])
    loss = F.cross_entropy(logits, target)
    
    return loss

# 测试
V, d = 10, 4
W_in = torch.randn(V, d)
W_out = torch.randn(d, V)
context_indices = torch.tensor([1, 2, 4, 5])
target_index = 3

loss = cbow_forward_loss(context_indices, target_index, W_in, W_out)
print("CBOW Loss:", loss.item())

CBOW Loss: 5.906073093414307


## 6 注意力机制

### 6.1 理论计算题

**计算缩放点积注意力**：

给定：
$Q \in \mathbb{R}^{2 \times 4}, K \in \mathbb{R}^{3 \times 4}, V \in \mathbb{R}^{3 \times 5}, d_k = 4$

**步骤**：
1. **计算得分矩阵 $S$**：
   $S = \frac{QK^\top}{\sqrt{d_k}} = \frac{QK^\top}{2}$
   $S$ 的维度为 $(2 \times 3)$。

2. **计算注意力权重 $A$**：
   $A = \text{softmax}(S, \text{dim}=-1)$
   对 $S$ 的每一行进行 softmax，使得每行和为 1。

3. **计算输出 $O$**：
   $O = AV$
   $O$ 的维度为 $(2 \times 5)$。

### 6.2 编程题

实现多头注意力前向传播。

In [5]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, X):
        # X: (seq_len, batch, d_model)
        seq_len, batch, _ = X.shape
        
        # 1. 线性投影并分头
        # (seq_len, batch, num_heads, d_k) -> (batch, num_heads, seq_len, d_k)
        Q = self.W_q(X).view(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        K = self.W_k(X).view(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        V = self.W_v(X).view(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        
        # 2. 缩放点积注意力
        # scores: (batch, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = torch.softmax(scores, dim=-1)
        
        # 3. 加权求和
        # context: (batch, num_heads, seq_len, d_k)
        context = torch.matmul(attn, V)
        
        # 4. 拼接并过最终线性层
        # (batch, num_heads, seq_len, d_k) -> (seq_len, batch, d_model)
        context = context.permute(2, 0, 1, 3).contiguous().view(seq_len, batch, self.d_model)
        output = self.W_o(context)
        
        return output

# 测试
seq_len, batch, d_model, num_heads = 10, 2, 4, 2
X = torch.randn(seq_len, batch, d_model)
mha = MultiHeadAttention(d_model, num_heads)
output = mha(X)
print("MHA Output shape:", output.shape) # 应为 (10, 2, 4)

MHA Output shape: torch.Size([10, 2, 4])
